In [4]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from helpers import *
import json
from logics import *

MANUAL_RETURN_VALUE = -100000

# Load model
model = YOLO("/ssd1/tuannw/yolo/4batch3/weights/best.pt")
  

In [5]:
# Đường dẫn đến file JSON
file_path = '/ssd1/tuannw/counting_longle/missing_case (1).json'

# Đọc file JSON
with open(file_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)


In [6]:
# Define class-specific colors
color_map = {
    0: (255, 0, 0),   # Red (in RGB)
    1: (0, 0, 255),   # Blue (in RGB)
}

PASS_CASE = []
FAILED_CASE = []
MANUAL_CASE = []


top_face_class_idx = 0

for image_path in metadata.keys():
    # if image_path != '/ssd1/tuannw/output_fix_merged/[29-05-25]/ORAL - Double deep/Rack BE/BE - 040 - 1_.png':
    #     continue
    # ### Process front image
    
    #TEST MANUAL CASE
    # nums_miss_gt = metadata[image_path]["nums_miss"]
    # if nums_miss_gt != -1:
    #     continue
    
    print(image_path)
    if not os.path.exists(image_path):
        continue
    front_image_path = metadata[image_path]['front_image']
    results = model(front_image_path, verbose=False)
    result = results[0]
    boxes = result.boxes.data.cpu().numpy().tolist()

    boxes = sorted(boxes, key=lambda x: -x[3])
    boxes = np.array(boxes)
    # 0: front face
    # 1: top face
    positive_boxes = boxes[boxes[:, -1] == 1 - top_face_class_idx]
    negative_boxes = boxes[boxes[:, -1] == top_face_class_idx]

    img_front = cv2.imread(front_image_path)

    layers_front = {}
    layer_idx = 1
    for bbox in positive_boxes:
        if f"layer_{layer_idx}" not in layers_front:
            layers_front[f"layer_{layer_idx}"] = [bbox]
            continue

        y_cur = (bbox[1] + bbox[3]) / 2
        y_cluster_center = sum([(bbox[1] + bbox[3]) / 2 for bbox in layers_front[f"layer_{layer_idx}"]]) / len(layers_front[f"layer_{layer_idx}"])
        avg_positive_boxes_height = sum([(bbox[3] - bbox[1]) for bbox in layers_front[f"layer_{layer_idx}"]]) / len(layers_front[f"layer_{layer_idx}"])

        if abs(y_cur - y_cluster_center) > avg_positive_boxes_height * 0.5:
            layer_idx += 1
            layers_front[f"layer_{layer_idx}"] = [bbox]
            continue
        else:
            layers_front[f"layer_{layer_idx}"].append(bbox)


    num_layers_front = len(layers_front)

    ### Process top down image
    # Chạy inference
    results = model(image_path, verbose=False, conf=0.5, iou=0.5)
    result = results[0]
    # Lấy bounding boxes
    masks = result.masks.xy if result.masks is not None else None
    boxes = result.boxes.data.cpu().numpy()
    boxes, masks = remove_overlapping_boxes(boxes, masks, ioa_thresh=0.7)
    # boxes = remove_nested_boxes(boxes, threshold=0.8)
    # boxes_copy = boxes.copy()
    # masks_copy = masks.copy()

    
    img = cv2.imread(image_path)

    # boxes = sorted(boxes, key=lambda x: -x[3])
    # boxes = np.array(boxes)
    # 0: front face
    # 1: top face

    box_mask_pairs = list(zip(boxes, masks))
    
    
    front_face_corners = []
    top_face_corners = []
    for box, mask in zip(boxes, masks):
        if int(box[5]) == 1:
            front_face_corners.append(find_corners(mask))
        elif int(box[5]) == 0:
            top_face_corners.append(find_corners(mask))
    matched_top, matched_front = find_top_for_front(front_face_corners, top_face_corners, threshold=200)
    
    is_between = -1
    # Lấy danh sách các bottom corners từ matched_front (giả sử thứ tự: [top-left, top-right, bottom-right, bottom-left])
    # front_bottom_corners = []
    # for poly in matched_front:
    #     # bottom-right và bottom-left
    #     front_bottom_corners.append(tuple(poly[2]))
    #     front_bottom_corners.append(tuple(poly[3]))

    # # Lấy tất cả các điểm từ matched_top
    # top_points = set()
    # for poly in matched_top:
    #     for pt in poly:
    #         top_points.add(tuple(pt))

    # # Kiểm tra bottom-left của matched_front với top-left của matched_top (nâng cao)
    # def is_chain_matched(front_polys, top_polys, threshold=100, depth=2):
    #     """
    #     Recursively check if bottom-left of a matched front matches top-left of any matched top,
    #     or bottom-right matches top-right, and continue for the matched top's corresponding front, up to a certain depth.
    #     """
    #     if depth == 0 or not front_polys or not top_polys:
    #         return False
    #     for front_poly in front_polys:
    #         front_bottom_left = tuple(front_poly[3])
    #         front_bottom_right = tuple(front_poly[2])
    #         for top_poly in top_polys:
    #             top_top_left = tuple(top_poly[0])
    #             top_top_right = tuple(top_poly[1])
    #             if euclidean(front_bottom_left, top_top_left) < threshold or euclidean(front_bottom_right, top_top_right) < threshold:
    #                 next_front = [top_poly]
    #                 next_tops = [tp for tp in top_polys if tp != top_poly]
    #                 # Nếu depth-1 == 0 thì đã kiểm tra đủ số lần, trả về True
    #                 if depth-1 == 0:
    #                     return True
    #                 if is_chain_matched(next_front, next_tops, threshold, depth-1):
    #                     return True
    #     return False

    # if is_chain_matched(matched_front, matched_top, 50, depth=2):
    #     is_between = 1



    # Sort based on box[3] (e.g., y2 coordinate) descending
    box_mask_pairs_sorted = sorted(box_mask_pairs, key=lambda x: -x[0][3])
    boxes, masks = zip(*box_mask_pairs_sorted)
    boxes = np.array(boxes)
    
    boxes_copy = boxes.copy()
    masks_copy = list(masks).copy()


    positive_boxes = boxes[boxes[:, -1] == 1 - top_face_class_idx]
    negative_boxes = boxes[boxes[:, -1] == top_face_class_idx]

    # ADD HERE
    #
    positive_indices = boxes_copy[:, -1] == 1 - top_face_class_idx
    negative_indices = boxes_copy[:, -1] == top_face_class_idx


    positive_masks = [masks_copy[idx] for idx, v in enumerate(positive_indices) if v == True ]
    #


    num_front_boxes = len(layers_front[f"layer_{len(layers_front)}"])
    min_y_distance_top_face = np.min([n_box[3] - n_box[1] for n_box in negative_boxes[:num_front_boxes]])

    layers = {}
    layers_mask = {}
    layer_idx = 1
    for idx, bbox in enumerate(positive_boxes):
        if f"layer_{layer_idx}" not in layers:
            layers[f"layer_{layer_idx}"] = [bbox]
            layers_mask[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue

        y_cur = (bbox[1] + bbox[3]) / 2
        y_cluster_center = sum([(bbox[1] + bbox[3]) / 2 for bbox in layers[f"layer_{layer_idx}"]]) / len(layers[f"layer_{layer_idx}"])
        avg_positive_boxes_height = sum([(bbox[3] - bbox[1]) for bbox in layers[f"layer_{layer_idx}"]]) / len(layers[f"layer_{layer_idx}"])

        if abs(y_cur - y_cluster_center) > avg_positive_boxes_height * 0.6:
            layer_idx += 1
            layers[f"layer_{layer_idx}"] = [bbox]
            layers_mask[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue
        else:
            layers[f"layer_{layer_idx}"].append(bbox)
            layers_mask[f"layer_{layer_idx}"].append(positive_masks[idx])

    num_layers = len(layers)

    # for key, value in layers.items():
    #     print(key, "=====> ", len(value))

    # num_layers = len(layers)
    # print("Number of layers:", num_layers)

    rule = []

    # # Check if positive boxes in last layer are equal to positive boxes in front face
    is_missing = False
    if len(layers_front[f"layer_{num_layers_front}"]) != len(layers[f"layer_{num_layers}"]):
        is_missing = True
        rule.append(1)

    if num_layers > 1:
        last_layer_boxes = layers[f"layer_{num_layers}"]
        pre_last_layer_boxes = layers[f"layer_{num_layers - 1}"]
        # Sort theo y_max, ưu tiên các box ở ngoài cùng
        last_layer_boxes = sorted(last_layer_boxes, key=lambda x: x[3], reverse=True)
        pre_last_layer_boxes = sorted(pre_last_layer_boxes, key=lambda x: x[3], reverse=True)

        # avg_height_last_layer = sum([(bbox[3] - bbox[1]) for bbox in last_layer_boxes]) / len(last_layer_boxes)
        box1 = last_layer_boxes[0]
        box2 = pre_last_layer_boxes[0]
        # if abs(box1[3] - box2[1]) > avg_height_last_layer * 0.3:
        #     is_missing = True
        if abs(box1[3] - box2[1]) > min_y_distance_top_face * 0.6:
            is_missing = True
            rule.append(2)


    # Kiểm tra độ dài x, sau này chuyển qua dựa trên pallet
    simulate_pallet_front = -1
    simulate_pallet_top = -1
    # max_distance_top_idx = -1
    max_distance_front_idx = -1

    for idx, f_layer in enumerate(layers_front):
        last_x_min, last_x_max = get_x_min_x_max(layers_front[f_layer])
        if abs(last_x_max - last_x_min) > simulate_pallet_front:
            simulate_pallet_front = abs(last_x_max - last_x_min)
            max_distance_front_idx = idx

    for idx, t_layer in enumerate(layers):
        last_x_min, last_x_max = get_x_min_x_max(layers[t_layer])
        if abs(last_x_max - last_x_min) > simulate_pallet_top:
            simulate_pallet_top = abs(last_x_max - last_x_min)
            # max_distance_top_idx = idx

    if num_layers_front > 1:
        last_layer_front_boxes = np.array(layers_front[f"layer_{num_layers_front}"])

        # Tính x_min và x_max
        last_x_min, last_x_max = get_x_min_x_max(last_layer_front_boxes)
        last_x_dis = abs(last_x_max - last_x_min)
        
        if 1 * last_x_dis < simulate_pallet_front * 0.9:
            is_missing = True
            rule.append(3)
    
    if num_layers > 1:
        last_layer_top_boxes = np.array(layers[f"layer_{num_layers}"])

        # Tính x_min và x_max
        last_x_min, last_x_max = get_x_min_x_max(last_layer_top_boxes)
        last_x_dis = abs(last_x_max - last_x_min)
        
        if 1 * last_x_dis < simulate_pallet_top * 0.9:
            is_missing = True
            rule.append(3)

    image = cv2.imread(image_path)
    if is_missing == False:
        is_missing = check_missing_by_area(image, boxes_copy, masks_copy, len(layers_front[f"layer_{len(layers_front)}"]), top_face_class_idx)
        if is_missing:
            rule.append(4)
    # else:
    #     if rule == [1]:
    #         is_missing = check_missing_by_area(image, boxes_copy, masks_copy, len(layers_front[f"layer_{len(layers_front)}"]))


    negative_indices = boxes_copy[:, -1] == top_face_class_idx
 
    negative_boxes = boxes_copy[negative_indices]
 
    n_masks = [masks_copy[idx] for idx, v in enumerate(negative_indices) if v == True]
 
    farthest_distance_top_face, _ = compute_extreme_x_distance(n_masks)

    last_layer_top_boxes = []
    last_layer_top_masks = []
    max_distance_top_idx = -1
    max_distance_top_layer = -1

    for idx, t_layer in enumerate(layers):
        total_covered, _, _ = compute_covered_x_distance(layers[t_layer])
        if total_covered > max_distance_top_layer:
            max_distance_top_layer = total_covered
            max_distance_top_idx = idx

    if max_distance_top_layer < 0.85 * farthest_distance_top_face:
        max_distance_top_layer = farthest_distance_top_face
        max_distance_top_idx -= 1 

    for idx, t_layer in enumerate(layers):
        if idx <= max_distance_top_idx:
            continue

        # total_covered, _, _ = compute_covered_x_distance(layers[f"layer_{idx + 1}"])
        try:
            total_covered, _, _ = compute_covered_x_distance_from_polygons(layers_mask[f"layer_{idx + 1}"])
        except:
            print('error',image_path)
            MANUAL_CASE.append([image_path])
            continue
        
        # print(total_covered / max_distance_top_layer)
        if 1 * total_covered < max_distance_top_layer * 0.9:
            last_layer_top_boxes.extend(layers[f"layer_{idx + 1}"])
            last_layer_top_masks.extend(layers_mask[f"layer_{idx + 1}"])

    if len(layers) >= 2:
        if max_distance_top_idx == len(layers) - 2:    
            if check_spacing_between_boxes(layers[f"layer_{max_distance_top_idx + 1}"], threshold=0.6) == False:
                last_layer_top_boxes.extend(layers[f"layer_{max_distance_top_idx + 1}"])
                last_layer_top_masks.extend(layers_mask[f"layer_{max_distance_top_idx + 1}"])


    if len(last_layer_top_boxes) == 0:
        last_layer_top_boxes = layers[f"layer_{num_layers}"]
        last_layer_top_masks = layers_mask[f"layer_{num_layers}"]

    assert len(last_layer_top_boxes) == len(last_layer_top_masks)

    positive_indices = boxes_copy[:, -1] == 1 - top_face_class_idx
    negative_indices = boxes_copy[:, -1] == top_face_class_idx

    positive_boxes = boxes_copy[positive_indices]
    negative_boxes = boxes_copy[negative_indices]

    p_masks = [masks_copy[idx] for idx, v in enumerate(positive_indices) if v == True ]
    n_masks = [masks_copy[idx] for idx, v in enumerate(negative_indices) if v == True]

    try:
        filtered_negative_boxes, filtered_negative_masks, remove_negative_masks, remove_negative_boxes = filter_negative_boxes(negative_boxes, n_masks, threshold=0.7)
    except:
        filtered_negative_boxes, filtered_negative_masks, remove_negative_masks, remove_negative_boxes = negative_boxes, n_masks, [], []


    # print('last layer top boxes:',last_layer_top_boxes)
    try:
        filter_data = np.concatenate((last_layer_top_boxes, filtered_negative_boxes), axis=0)
        filter_data_masks = last_layer_top_masks + filtered_negative_masks
    except:
        print("error: ", image_path)
        FAILED_CASE.append([image_path])
        continue
    
    filter_data = convert_bbox_xyxy_to_xywh(filter_data)
    filter_data, filtered_data_ingore, filtered_data_masks_accept, filtered_data_masks_ignore, bx, by, num_front_face_in_line, filtered_data_class_ids_accept = interpolate_boundary(filter_data, filter_data_masks, target_class_idx=top_face_class_idx)
   
    filter_data = np.array(filter_data)
    filter_data = filter_data_below_front_face(filter_data, threshold=0.9)
    print("\n", filter_data)
    #New here
    
    top_face_masks_accept = [mask for mask, class_id in zip(filtered_data_masks_accept, filtered_data_class_ids_accept) if class_id == top_face_class_idx]
    
    front_face_masks_accept = [mask for mask, class_id in zip(filtered_data_masks_accept, filtered_data_class_ids_accept) if class_id == 1 - top_face_class_idx]
    
    all_topface_masks = top_face_masks_accept + filtered_data_masks_ignore

    top_corners = []
    for i, mask in enumerate(all_topface_masks):
        corners = find_corners(mask)  
        top_corners.append(corners)

    #case thiếu
    # matched_top_face_corners = [find_corners(mask) for mask in top_face_masks_accept]

    # top_face_stack = find_all_upper_layers_by_mask(matched_top_face_corners,top_corners, threshold=30)

    # #case dư
    front_face_corners_accept=[find_corners(front) for front in front_face_masks_accept]
    matched_top_face_corners, _ = find_top_for_front(front_face_corners_accept, top_corners, threshold=200)

    top_face_stack = find_all_upper_layers_by_mask(matched_top_face_corners,top_corners, threshold=30)
    
    nums_miss = len(top_face_stack)

    
    angles = []
    for box in top_face_stack:
        _, _, angle = compute_box_slope(box)
        angles.append(angle)
        
    manual_check = False
    for slope in angles:
        if slope > 12:
            manual_check = True

    # nums_miss = sum(1 for obj in filter_data if obj[5] == top_face_class_idx)
    num_class_pos = sum(1 for obj in filter_data if obj[5] == 1 - top_face_class_idx)
    print("num_class_pos:", num_class_pos)
    print("num_front_face_in_line:", num_front_face_in_line)
    print("nums_miss:", nums_miss)
    if nums_miss < num_class_pos:
        nums_miss = num_class_pos
    if (num_class_pos != num_front_face_in_line) or manual_check:
        # print('num_class_pos',num_class_pos)
        # print('num front face in line:',num_front_face_in_line)
        nums_miss = MANUAL_RETURN_VALUE



    # plot_filtered_data(filter_data, bx, by)

    # if 1 in rule:
    #     nums_miss = MANUAL_RETURN_VALUE

    ground_truth_missing = metadata[image_path]['missing']
    nums_miss_gt = metadata[image_path]["nums_miss"]
    print
    
    if is_between == 1: 
        nums_miss = MANUAL_RETURN_VALUE
        
    if nums_miss != nums_miss_gt and nums_miss_gt != -1:
        if nums_miss != MANUAL_RETURN_VALUE:
            FAILED_CASE.append([image_path, rule])
        else:
            MANUAL_CASE.append([image_path])
    else:
        PASS_CASE.append([image_path])


    print("\n", nums_miss)

  

/ssd1/tuannw/output_fix_merged/[30-05-25]/FOOD - Double deep/Rack AZ/AZ - 001 - 1_.png


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
print("PASS")
for case in PASS_CASE:
    print(case)



PASS


In [ ]:
print(len(PASS_CASE))


0


In [ ]:
# 67 chỉ có rule lan
# 64 threshold 50
# 62 threshold 30

In [ ]:
# fail ,manual  : slope > 12,  thresh 30
# 58 
# 49 

In [ ]:
# fail ,manual  : slope > 12,  thresh 30, sửa num front in line
# 72
# 23

# 68
# 23

In [ ]:
# 55
# 39 thresh num -5 5 , output mới

In [ ]:
# 49  case dư  output mới
# 54

# 48  case thiếu output mới
# 55

In [ ]:

# 51 case dư output cũ
# 54

# 50 case thiếu output cũ
# 55


In [ ]:
# 44 case dư output mới
# 53

# 46 case thiếu output mới
# 53

In [ ]:

# 49 case dư output cũ
# 53

# 49 case thiếu output cũ
# 53




In [ ]:
# code a longle
# 55
# 46
# code c thaokb (-7/+0)
# 47
# 46


# 55
# 48
# fix lan + rule interpolate + sort_corners + filter_negative_boxes

# 41
# 53
# fix lan + rule interpolate + sort_corners + filter_negative_boxes + add smoothing mask


In [ ]:
print("=" * 25)
print("FAILED")
for case, rule in FAILED_CASE:
    print(case)

print("=" * 25)
print("MANUAL")
for case in MANUAL_CASE:
    print(case)

FAILED
/ssd1/tuannw/output_fix_merged/[30-05-25]/FOOD - Double deep/Rack BA/BA - 097 - 1_.png
/ssd1/tuannw/output_fix_merged/[30-05-25]/FOOD - Double deep/Rack BB/BB - 094 - 1_.png
/ssd1/tuannw/output_fix_merged/[29-05-25]/PC/Rack AQ/AQ - 106 - 1.png
/ssd1/tuannw/output_fix_merged/[29-05-25]/PC/Rack AS/AS - 088 - 1_.png
/ssd1/tuannw/output_fix_merged/[27-05-25]/ORAL/Single deep/Rack AW/AW - 094 - 1_.png
/ssd1/tuannw/output_fix_merged/[27-05-25]/PC/Rack AL/AL - 120 - 1_.png
/ssd1/tuannw/output_fix_merged/[27-05-25]/PC/Rack AJ/AJ - 095 - 1_.png
/ssd1/tuannw/output_fix_merged/[27-05-25]/PC/Rack AJ/AJ - 031 - 1_.png
/ssd1/tuannw/output_fix_merged/[28-05-25]/FOOD/Double deep/Rack BA/BA - 090 - 1_.png
/ssd1/tuannw/output_fix_merged/[28-05-25]/FOOD/Double deep/Rack BB/BB - 027 - 1_.png
/ssd1/tuannw/output_fix_merged/[28-05-25]/ORAL/Rack AV/AV - 033 - 1_.png
/ssd1/tuannw/output_fix_merged/[28-05-25]/ORAL/Rack AV/AV - 023 - 1_.png
/ssd1/tuannw/output_fix_merged/[28-05-25]/ORAL/Rack AU/AU - 014 

In [ ]:

print(len(FAILED_CASE))
print(len(MANUAL_CASE))

28
26
